# Assignment 11: Fine-Tuning GPT-2 on a Custom Text Style

**Author:** Hamzah Tareq Al-Qudaimi, BCA, Alliance University

**Goal:** show how fine-tuning makes a pretrained language model adopt a specific writing style.

| Assignment step | Where in this notebook |
|---|---|
| Collect ~200-300 lines in a consistent style | Section 1 (`raw_dataset.txt`, 240 dialogue lines) |
| Clean the dataset (timestamps, usernames, formatting, non-language content) | Section 2 |
| Generate from the **original** GPT-2 with the Assignment 2 prompt | Section 3 |
| Fine-tune small GPT-2 with the Hugging Face `Trainer` | Section 4 |
| Generate from the **fine-tuned** model with the same prompt | Section 5 |
| Compare before and after, identify at least two style differences | Section 6 |

**How to run (Google Colab):** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**. 
When asked, upload `raw_dataset.txt`. Total run time is roughly 5 minutes.

**Before you run:** set `PROMPT` in the next cell to the exact prompt you used in Assignment 2.

## 0. Setup and configuration

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import os, re, math, random, inspect, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer,
                          TrainingArguments, set_seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "(a GPU makes training much faster)" if device == "cpu" else "")

# ----------------------------------------------------------------------------
# >>> SET THIS to the exact prompt you used with the original GPT-2 in Assignment 2 <<<
PROMPT = "REPLACE WITH YOUR ASSIGNMENT 2 PROMPT"
# ----------------------------------------------------------------------------
if PROMPT.startswith("REPLACE"):
    raise ValueError("Set PROMPT to your Assignment 2 prompt before running.")

SEED = 42
MODEL_NAME = "gpt2"          # the small GPT-2 (124M parameters)
BLOCK_SIZE = 64              # tokens per training example
EPOCHS = 10
BATCH_SIZE = 4
LEARNING_RATE = 5e-5

# Generation settings (use the same ones you used in Assignment 2 if they differ)
GEN = dict(max_new_tokens=80, do_sample=True, top_k=50, top_p=0.95, temperature=0.8)
N_SAMPLES = 5

set_seed(SEED)

## 1. The dataset

`raw_dataset.txt` is a chat log of two fictional characters on a small cargo ship: **Mara** (the navigator) and **Teo** (the cook). 
The writing style is consistent throughout: short sentences, deadpan understatement, nautical and food metaphors, and a quip at the end of most lines. 
It has 240 dialogue lines, written as an original text for this assignment. 

Like a real exported chat, the file also contains **timestamps, usernames, system messages, attachment markers, URLs, emojis, and stray formatting**, which must be removed before training.

In [ ]:
RAW_PATH = "raw_dataset.txt"
if not os.path.exists(RAW_PATH):
    from google.colab import files
    print("Upload raw_dataset.txt")
    files.upload()

with open(RAW_PATH, encoding="utf-8") as f:
    raw_lines = f.read().splitlines()
print("Total lines in raw file:", len(raw_lines))
print()
print("\n".join(raw_lines[:12]))

## 2. Cleaning the dataset

Each raw line is cleaned by these rules:

1. Remove **timestamps** like `[2026-03-14 02:14:54]`.
2. Drop **system / non-language lines**: `*** ... ***` notices, `[system] ...`, `<<attachment: ...>>`, bare URLs, blank lines.
3. Replace **usernames** (`@mara_nav:`) with a plain speaker label (`Mara:`), so the model learns a dialogue format instead of memorising account names.
4. Remove inline **URLs** and leftover formatting such as `(edited)`.
5. Remove **emojis / non-ASCII symbols**.
6. Collapse repeated whitespace.

In [ ]:
USERNAMES = {"mara_nav": "Mara", "teo_galley": "Teo"}

TIMESTAMP   = re.compile(r"^\s*\[\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\]\s*")
USER_PREFIX = re.compile(r"^@(\w+):\s*(.*)$")
SEE_URL     = re.compile(r"\bsee\s+https?://\S+")
URL         = re.compile(r"https?://\S+")

def clean_line(raw):
    line = TIMESTAMP.sub("", raw)                                   # 1. timestamps
    if not line.strip():                                            # 2. blank lines
        return None
    if line.startswith(("***", "[system]", "<<attachment")):        # 2. system / attachment lines
        return None
    m = USER_PREFIX.match(line)
    if not m:                                                       # 2. anything that is not a chat message (e.g. bare URL)
        return None
    user, text = m.groups()
    text = SEE_URL.sub("", text)                                    # 4. inline urls
    text = URL.sub("", text)
    text = text.replace("(edited)", "")                             # 4. formatting leftovers
    text = text.encode("ascii", "ignore").decode()                  # 5. emojis / non-ASCII
    text = re.sub(r"\s+", " ", text).strip()                        # 6. whitespace
    if not text:
        return None
    speaker = USERNAMES.get(user, user.title())                     # 3. usernames -> speaker labels
    return f"{speaker}: {text}"

clean_lines = [c for c in (clean_line(l) for l in raw_lines) if c]

print(f"Raw lines: {len(raw_lines)}  ->  clean dialogue lines: {len(clean_lines)}")
print()
print("BEFORE:", raw_lines[5])
print("AFTER: ", clean_line(raw_lines[5]))
print()
print("BEFORE:", raw_lines[8])
print("AFTER: ", clean_line(raw_lines[8]))

# sanity checks: nothing unwanted survived
bad = [l for l in clean_lines if re.search(r"https?://|@\w+|\[\d{4}|\*\*\*|<<|[^\x00-\x7f]|\s{2,}", l)]
assert not bad, bad[:3]
print("\nCleaning check passed: no timestamps, usernames, URLs, emojis or double spaces remain.")

with open("cleaned_dataset.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(clean_lines) + "\n")
TRAIN_VOCAB = set(re.findall(r"[A-Za-z']+", " ".join(clean_lines).lower()))
print("Vocabulary size of the cleaned dataset:", len(TRAIN_VOCAB), "unique words")

## 3. Base model output (original GPT-2, before fine-tuning)

The prompt is run through the **untouched** pretrained GPT-2. Every model gets the same prompt, same sampling settings and same random seed.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
print("Parameters:", sum(p.numel() for p in base_model.parameters()) / 1e6, "million")

def generate(model, prompt, n=N_SAMPLES, seed=SEED):
    model.eval()
    set_seed(seed)
    inputs = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, num_return_sequences=n, pad_token_id=tok.eos_token_id, **GEN)
    return [tok.decode(o, skip_special_tokens=True) for o in out]

base_outputs = generate(base_model, PROMPT)
for i, o in enumerate(base_outputs, 1):
    print(f"--- BASE sample {i} ---\n{o}\n")

with open("base_output.txt", "w", encoding="utf-8") as f:
    f.write(f"PROMPT: {PROMPT}\nMODEL: original {MODEL_NAME}\nSETTINGS: {GEN}, seed={SEED}\n\n")
    for i, o in enumerate(base_outputs, 1):
        f.write(f"--- sample {i} ---\n{o}\n\n")

## 4. Fine-tuning with the Hugging Face `Trainer`

**Preparing the data.** The cleaned lines are tokenised (each line ends with a newline token), joined into one long token stream, and cut into blocks of 64 tokens. 
About 10% of the blocks are held out as an **evaluation set** (never trained on) so we can measure how well each model predicts the target style.

**Training.** Causal language modelling: the model learns to predict each next token, with labels equal to the inputs. 
The held-out loss of the base model is measured first, so we can see how much fine-tuning improves it.

In [ ]:
# ---- tokenise and cut into blocks
ids = []
for line in clean_lines:
    ids += tok(line + "\n")["input_ids"]
blocks = [ids[i:i + BLOCK_SIZE] for i in range(0, len(ids) - BLOCK_SIZE + 1, BLOCK_SIZE)]
random.Random(SEED).shuffle(blocks)
n_eval = max(2, round(0.1 * len(blocks)))
eval_blocks, train_blocks = blocks[:n_eval], blocks[n_eval:]
print(f"Total tokens: {len(ids)} | blocks of {BLOCK_SIZE}: {len(blocks)} "
      f"| train blocks: {len(train_blocks)} | eval blocks: {len(eval_blocks)}")

class BlockDataset(torch.utils.data.Dataset):
    def __init__(self, blocks): self.blocks = blocks
    def __len__(self): return len(self.blocks)
    def __getitem__(self, i):
        x = torch.tensor(self.blocks[i])
        return {"input_ids": x, "labels": x.clone()}      # causal LM: predict the next token

train_ds, eval_ds = BlockDataset(train_blocks), BlockDataset(eval_blocks)

# transformers renamed `evaluation_strategy` to `eval_strategy`; support both
EVAL_KEY = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"

def make_args():
    return TrainingArguments(
        output_dir="gpt2-style-checkpoints",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
        **{EVAL_KEY: "epoch"},
    )

# ---- held-out loss of the BASE model (before any training)
base_eval = Trainer(model=base_model, args=make_args(), eval_dataset=eval_ds).evaluate()
base_loss = base_eval["eval_loss"]
print(f"Base GPT-2 held-out loss: {base_loss:.3f} | perplexity: {math.exp(base_loss):.1f}")

In [ ]:
# ---- fine-tune a fresh copy of GPT-2
set_seed(SEED)
ft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
trainer = Trainer(model=ft_model, args=make_args(), train_dataset=train_ds, eval_dataset=eval_ds)
trainer.train()

ft_eval = trainer.evaluate()
ft_loss = ft_eval["eval_loss"]
print(f"\nFine-tuned held-out loss: {ft_loss:.3f} | perplexity: {math.exp(ft_loss):.1f}")
print(f"Base model was at       : {base_loss:.3f} | perplexity: {math.exp(base_loss):.1f}")

In [ ]:
# ---- training curves
log = trainer.state.log_history
tr = [(l["epoch"], l["loss"]) for l in log if "loss" in l]
ev = [(l["epoch"], l["eval_loss"]) for l in log if "eval_loss" in l]
plt.figure(figsize=(6.5, 3.6))
plt.plot(*zip(*tr), label="training loss", color="#2C6FBB")
plt.plot(*zip(*ev), "o-", label="held-out loss", color="#D1495B")
plt.axhline(base_loss, ls="--", color="grey", label="base GPT-2 held-out loss")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss"); plt.legend(frameon=False); plt.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 5. Fine-tuned model output (same prompt, same settings, same seed)

In [ ]:
ft_model.to(device)
ft_outputs = generate(ft_model, PROMPT)
for i, o in enumerate(ft_outputs, 1):
    print(f"--- FINE-TUNED sample {i} ---\n{o}\n")

with open("finetuned_output.txt", "w", encoding="utf-8") as f:
    f.write(f"PROMPT: {PROMPT}\nMODEL: {MODEL_NAME} fine-tuned for {EPOCHS} epochs\nSETTINGS: {GEN}, seed={SEED}\n\n")
    for i, o in enumerate(ft_outputs, 1):
        f.write(f"--- sample {i} ---\n{o}\n\n")

## 6. Before-and-after comparison

First, the outputs are compared with simple measurable style features. Then you write the analysis.

In [ ]:
def continuation(text, prompt=PROMPT):
    return text[len(prompt):] if text.startswith(prompt) else text

def style_metrics(texts):
    text = "\n".join(continuation(t) for t in texts)
    words = re.findall(r"[A-Za-z']+", text.lower())
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    dialogue = sum(bool(re.match(r"^(Mara|Teo):", l)) for l in lines)
    return {
        "Total words generated": len(words),
        "Average words per sentence": round(len(words) / max(1, len(sentences)), 1),
        "Average words per line": round(len(words) / max(1, len(lines)), 1),
        "% of lines in 'Name: ...' dialogue format": round(100 * dialogue / max(1, len(lines)), 1),
        "% of words that appear in the training data": round(100 * sum(w in TRAIN_VOCAB for w in words) / max(1, len(words)), 1),
        "Unique words / total words": round(len(set(words)) / max(1, len(words)), 2),
    }

cmp_df = pd.DataFrame({"Base GPT-2": style_metrics(base_outputs),
                       "Fine-tuned GPT-2": style_metrics(ft_outputs)})
cmp_df.loc["Held-out perplexity on the target style"] = [round(math.exp(base_loss), 1), round(math.exp(ft_loss), 1)]
cmp_df.to_csv("comparison_metrics.csv")
cmp_df

In [ ]:
# ---- side-by-side before/after file for submission
lines = [f"# Before-and-after comparison\n", f"**Prompt:** {PROMPT}\n",
         f"**Settings:** {GEN}, seed={SEED}, {N_SAMPLES} samples per model\n", "## Base GPT-2 output\n"]
for i, o in enumerate(base_outputs, 1):
    lines += [f"**Sample {i}**\n", "```", o, "```\n"]
lines += ["## Fine-tuned GPT-2 output\n"]
for i, o in enumerate(ft_outputs, 1):
    lines += [f"**Sample {i}**\n", "```", o, "```\n"]
lines += ["## Measured style features\n", "| Feature | Base GPT-2 | Fine-tuned GPT-2 |", "|---|---|---|"]
for feat, row in cmp_df.iterrows():
    lines.append(f"| {feat} | {row['Base GPT-2']} | {row['Fine-tuned GPT-2']} |")
with open("before_after_comparison.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("Saved before_after_comparison.md")

# quick visual side by side for the first sample
print("=" * 30, "BASE (sample 1)", "=" * 30);       print(base_outputs[0])
print("=" * 28, "FINE-TUNED (sample 1)", "=" * 28);  print(ft_outputs[0])

### Analysis: style differences between the base model and the fine-tuned model

> **Write this section yourself after reading the actual outputs above.** The numbers and examples must come from *your* run, because sampling results differ between runs and machines. 
> Copy short quotes from `base_output.txt` and `finetuned_output.txt` as evidence.

**Difference 1: (e.g. format / structure)**
- Base model: ...
- Fine-tuned model: ...
- Evidence (quote + metric from the table above): ...

**Difference 2: (e.g. sentence length and rhythm)**
- Base model: ...
- Fine-tuned model: ...
- Evidence: ...

**Difference 3 (optional): (e.g. vocabulary and tone, such as nautical or food imagery, deadpan humour)**
- Base model: ...
- Fine-tuned model: ...
- Evidence: ...

**Things to look for:**
- *Format:* does the fine-tuned model produce `Mara:` / `Teo:` dialogue turns while the base model writes running prose?
- *Sentence length:* the training lines are short (about 8-12 words); are the fine-tuned sentences shorter? (see the metrics table)
- *Vocabulary and tone:* does the fine-tuned model reuse words and imagery from the dataset (fog, soup, ropes, stars) and the deadpan joking tone?
- *Content:* does the fine-tuned output stay on topic, or does it sometimes copy whole phrases from the training lines (a sign of memorisation)?

**Limitations to mention:** the dataset is tiny (240 lines), so the model learns the *style* quickly but can also over-fit and repeat training phrases; 
the comparison uses only one prompt and five samples; sampling is random, so a different seed would give different text.

## 7. Save the fine-tuned model checkpoint and the submission files

In [ ]:
SAVE_DIR = "gpt2-style-finetuned"
trainer.save_model(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)
shutil.make_archive("gpt2-style-finetuned", "zip", SAVE_DIR)

submission = ["raw_dataset.txt", "cleaned_dataset.txt", "base_output.txt", "finetuned_output.txt",
              "before_after_comparison.md", "comparison_metrics.csv", "gpt2-style-finetuned.zip"]
for f in submission:
    print(("OK      " if os.path.exists(f) else "MISSING ") + f)

try:
    from google.colab import files
    for f in submission:
        if os.path.exists(f): files.download(f)
except ImportError:
    print("Not running in Colab: files are in the current folder.")

## Submission checklist

| Required item | File |
|---|---|
| Training notebook or fine-tuned model checkpoint | this notebook and `gpt2-style-finetuned.zip` |
| Original dataset | `raw_dataset.txt` (and `cleaned_dataset.txt`) |
| Base-model output | `base_output.txt` |
| Fine-tuned model output | `finetuned_output.txt` |
| Before-and-after comparison | `before_after_comparison.md` and the analysis section above |

### Loading the saved model later
```python
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained("gpt2-style-finetuned")
model = AutoModelForCausalLM.from_pretrained("gpt2-style-finetuned")
```